In [1]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
%load_ext autoreload
#%load_ext line_profiler
#%load_ext snakeviz
%autoreload 2

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import corner

import pickle

import enterprise
from enterprise.pulsar import Pulsar
import enterprise.signals.parameter as parameter
from enterprise.signals import utils
from enterprise.signals import signal_base
from enterprise.signals import selections
from enterprise.signals.selections import Selection
from enterprise.signals import white_signals
from enterprise.signals import gp_signals
from enterprise.signals import deterministic_signals
import enterprise.constants as const

from enterprise_extensions import deterministic

from scipy.stats import norm

import libstempo as T2
import libstempo.toasim as LT
import libstempo.plot as LP

import glob
import json
import h5py
import healpy as hp
import scipy.constants as sc
import emcee

from numba.typed import List

import sys
import h5py

In [2]:
base = "IPTA_MDC2_G2D2_broad_detection_dL_75.4_"
crnr_plt_title1 =base+'corner plot_all_params'
crnr_plt_title2 =base
trace_plot_title = base+'trace_plot'
BF_histogram_title = base+'log10A_histogram'
BFerr_title = base+'log10A_histogram_error'

In [3]:
pwd

'/scratch/na00078/projects/IPTA_MDC2/post_processing'

## Generate Outfile

## Run Outfile

In [4]:
#COMMENT IF OUTFILE GENERATED
#Thinning script for one source
i = "/scratch/na00078/projects/IPTA_MDC2/h5_files/G2D2_detect_allsky_outfile.h5"
infile = i
first_n_param = 8
# outfile = '/scratch/na00078/QuickCW_targeted_runs/results/3C66B_outfile.h5'

print(infile)
print(first_n_param)

with h5py.File(infile, 'r') as f:
    Ts = f['T-ladder'][...]
    samples_cold = f['samples_cold'][:,:,:first_n_param]
    print(samples_cold[-1].shape)
    log_likelihood = f['log_likelihood'][:1,:]
    print(log_likelihood.shape)
    par_names = [x.decode('UTF-8') for x in list(f['par_names'])]
    acc_fraction = f['acc_fraction'][...]
    fisher_diag = f['fisher_diag'][...]

/scratch/na00078/projects/IPTA_MDC2/h5_files/G2D2_detect_allsky_outfile.h5
8
(100000000, 8)
(1, 100000000)


## Corner Plots

In [5]:
KPC2S = sc.parsec / sc.c * 1e3
SOLAR2S = sc.G / sc.c ** 3 * 1.98855e30


xxx0 = {"0_cos_gwtheta":np.cos(0.6387905062299246),
       "0_cos_inc":0.8412486994612669,
       "0_gwphi":3.3335788713091694,
       "0_log10_fgw":np.log10(3.7e-09),
       "0_log10_h":-13.668773493298787,
       "0_log10_mc":np.log10(4300000000),
       "phase0": 0.24434609527920614,
        "psi": 1.1187560505283651,
        "distance": 75.4,}


xxx = {
       "0_cos_inc":0.8412486994612669,
        "0_log10_fgw":np.log10(3.7e-09),
        "0_log10_h":-13.668773493298787,
       "0_log10_mc":np.log10(4300000000),
       "phase0": 0.24434609527920614,
        "psi": 1.1187560505283651}

xxx['gwb_gamma'] = np.nan    
xxx['gwb_log10_A'] = -15.070581074285707
    
#print(xxx)

# In[6]:




In [ ]:
import numpy as np
import corner

# ---------------------------------------------------------------------
# 1. Parameter keys including RA, Dec (derived from cos(theta), phi)
# ---------------------------------------------------------------------
# (We do NOT include log10_fgw here)
corner_mask = [0, 1, 2, 4, 5, 6, 7]   # drop index 3 (fgw)

par_keys = [
    "0_cos_gwtheta",    # 0
    "0_cos_inc",        # 1
    "0_gwphi",          # 2
    "0_log10_h",        # 4 (index in cold samples)
    "0_log10_mc",       # 5
    "0_phase0",         # 6
    "0_psi"             # 7
]

labels = [
    r"$\alpha\,$(RA)",          # NEW
    r"$\delta\,$(Dec)",         # NEW
    r"$\cos \iota$",
    r"$\log_{10} A_{\rm e}$",
    r"$\log_{10} {\cal M}$",
    r"$\Phi_0$",
    r"$\psi$"
]

label_to_key = {
    r"$\alpha\,$(RA)": "0_gwphi",
    r"$\delta\,$(Dec)": "0_cos_gwtheta",
    r"$\cos \iota$": "0_cos_inc",
    r"$\log_{10} A_{\rm e}$": "0_log10_h",
    r"$\log_{10} {\cal M}$": "0_log10_mc",
    r"$\Phi_0$": "phase0",
    r"$\psi$": "psi"
}

# ---------------------------------------------------------------------
# 2. Truth values
# ---------------------------------------------------------------------

truths = []
for l in labels:
    key = label_to_key[l]
    if l == r"$\delta\,$(Dec)":
        truths.append(np.degrees(np.arcsin(xxx0["0_cos_gwtheta"])))
    elif l == r"$\alpha\,$(RA)":
        truths.append(xxx0["0_gwphi"])
    else:
        truths.append(xxx0[key])

# ---------------------------------------------------------------------
# 3. Extract samples (NO dL MASKING)
# ---------------------------------------------------------------------
burnin = 0
thin = 1
samples_raw = samples_cold[0][burnin::thin, :]

cos_theta = samples_raw[:, 0]      # cos(gwtheta)
phi = samples_raw[:, 2]            # gwphi

# Compute RA & Dec
RA = phi
Dec = np.degrees(np.arcsin(cos_theta))

# Build plotting matrix WITHOUT fGW
samples2plot = np.vstack([
    RA,                       # RA
    Dec,                      # Dec
    samples_raw[:, 1],        # cos_inc
    samples_raw[:, 4],        # log10_h
    samples_raw[:, 5],        # log10_mc
    samples_raw[:, 6],        # phase0
    samples_raw[:, 7]         # psi
]).T

# ---------------------------------------------------------------------
# 4. Plot ranges (NO fGW)
# ---------------------------------------------------------------------
ranges = [
    (0, 2*np.pi),         # RA
    (-90, 90),            # Dec
    (-1, 1),              # cos i
    (-18, -11),           # log10 h
    (8, 10),              # log10 Mc
    (0, 2*np.pi),         # phase0
    (0, np.pi)            # psi
]

# ---------------------------------------------------------------------
# 5. Corner plot
# ---------------------------------------------------------------------
fig = corner.corner(
    samples2plot,
    labels=labels,
    truths=truths,
    truth_color="red",
    show_titles=True,
    range=ranges,
    hist_kwargs={"density": True}
)

for ax in fig.get_axes():
    ax.tick_params(axis="both", labelsize=14)
    ax.xaxis.label.set_size(16)
    ax.yaxis.label.set_size(16)

fig.suptitle("Corner plot (no $d_L$ masking, RA/Dec included, no $f_{\mathrm{GW}}$)", 
             fontsize=24, y=1.05)


## Trace Plots

In [ ]:
import matplotlib.pyplot as plt

title = ["gw cos_theta", "cos_inc", "gw phi", "log10 fGW", 
         "log10 h", "log10 Mc", "phase", "psi", "log10 dL"]

j = -1
row, column = 3, 3
fig, axs = plt.subplots(row, column, figsize=(30, 20))
fig.tight_layout(h_pad=3, w_pad=2)

# Loop over parameters
for i in range(len(title)): 
    if i % column == 0:
        j += 1
    axs[j, i % column].plot(d_L_mask, samples2plot[d_L_mask, i])
    axs[j, i % column].set_title(title[i], fontsize=30)    # bigger subplot title
    axs[j, i % column].tick_params(axis="both", labelsize=30)  # bigger tick labels
    axs[j, i % column].set_xlabel("sample index", fontsize=30)
    axs[j, i % column].set_ylabel("value", fontsize=30)

# Global title
fig.suptitle(trace_plot_title, fontsize=28, y=1.08)

plt.subplots_adjust(hspace=0.4, wspace=0.3)  # more spacing
#plt.show()
#plt.savefig("trace_plots.png", dpi=200, bbox_inches="tight")


In [ ]:
print(np.min(samples2plot[d_L_mask,3]))
print(np.max(np.log10(samples2plot[d_L_mask,5])))

## SD BF

In [ ]:
logAmin_dL = np.min(samples2plot[d_L_mask,4])
logAmax_dL = np.max(samples2plot[d_L_mask,4])
print("hmin",logAmin_dL)
print("hmax",logAmax_dL)
print("Mc min",np.min(samples2plot[d_L_mask,5]))
print("Mc max",np.max(samples2plot[d_L_mask,5]))

print(len(samples2plot[d_L_mask,4]))

plt.hist(samples2plot[d_L_mask,4], bins=20, density=True)
plt.hlines(1/(-11 - logAmin_dL), logAmin_dL, -11,  color='red')
plt.title(BF_histogram_title)
plt.show()

In [ ]:
#d_L correlated SD BF
logamins = np.linspace(logAmin_dL, -15, 100)
BF_arr = []
BF_err_arr = []

from enterprise_extensions import model_utils
for logamin in logamins:
    BF,BF_err = model_utils.bayes_fac(samples = samples2plot[d_L_mask,4], logAmin=logamin, logAmax=-11)
    BF_arr.append(BF)
    BF_err_arr.append(BF_err)

plt.errorbar(logamins, BF_arr, yerr=BF_err_arr, fmt='o')
plt.xlabel('logAmin')
plt.ylabel("log10(BF)")
plt.title(BFerr_title)
plt.show()

In [ ]:
#d_L correlated SD BF
BF, BF_err = model_utils.bayes_fac(samples=samples2plot[d_L_mask,4], logAmax=-11)
print(f"log10A BF = {BF:.4f} ± {BF_err:.4f}")

In [ ]:
#Non d_L correlated SD BF
from enterprise_extensions import model_utils
BF,BF_err = model_utils.bayes_fac(samples = samples_cold[0][::10,4], logAmax=-11)
print(f"log10A BF = {BF:.4f} ± {BF_err:.4f}")

In [ ]:
print(BF_arr[0:10])

In [ ]:
print(BF_err_arr[0:10])